# 02 · Models

Trains and evaluates all model families on the three selectivity tasks.
Baseline → diagnostics → tuned. All approaches use identical scaffold-split data from `data.pkl`.

| Approach | Target | Model |
|---|---|---|
| **A** — direct | ΔpChEMBL | RF / XGB / MLP |
| **B** — post-hoc | D2 and 5HT₂A separately, then subtract | RF / XGB / MLP × 2 |
| **C** — multi-task | D2 and 5HT₂A jointly | Two-headed MLP |

**Output:** `models.pkl`

## 0. Imports

In [1]:
import sys, pickle, warnings, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from utils import build_feature_matrix, fit_feature_pipeline, SEED, N_FP

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def seed_everything(seed=SEED):
    """Seed all sources of randomness for reproducibility."""
    random.seed(seed)                            # Python built-in
    os.environ['PYTHONHASHSEED'] = str(seed)     # Python hash randomisation
    np.random.seed(seed)                         # NumPy
    torch.manual_seed(seed)                      # PyTorch CPU
    torch.cuda.manual_seed_all(seed)             # PyTorch CUDA (no-op if absent)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)              # Apple MPS
    # Make PyTorch ops deterministic where possible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {DEVICE}  |  SEED: {SEED}")

Device: mps  |  SEED: 42


## 1. Load data

In [2]:
with open('data.pkl', 'rb') as f:
    data = pickle.load(f)

merged = data['merged']
d2     = data['d2']
sht    = data['sht']
sp     = data['splits']

smiles_ov  = merged['curated_smiles'].tolist()
smiles_d2  = d2['curated_smiles'].tolist()
smiles_sht = sht['curated_smiles'].tolist()

X_ov  = data['X_ov']
X_d2  = data['X_d2']
X_sht = data['X_sht']

y_del = merged['delta'].values
y_d2  = d2['pChEMBL'].values
y_sht = sht['pChEMBL'].values

tr_ov, va_ov, te_ov = sp['tr_ov'], sp['va_ov'], sp['te_ov']
tr_d2, va_d2, te_d2 = sp['tr_d2'], sp['va_d2'], sp['te_d2']
tr_st, va_st, te_st = sp['tr_sht'], sp['va_sht'], sp['te_sht']

# ── Fit feature pipelines on training splits ──────────────────────────────────
# fit_feature_pipeline fits VarianceThreshold on fingerprints and
# StandardScaler on the 9 physicochemical features — exactly what
# build_feature_matrix expects when called later.
from utils import fit_feature_pipeline

print("Fitting feature pipelines on training splits...")
X_feat_ov,  sel_ov,  sc_ov  = fit_feature_pipeline(X_ov[tr_ov],  [smiles_ov[i]  for i in tr_ov])
X_feat_d2,  sel_d2,  sc_d2  = fit_feature_pipeline(X_d2[tr_d2],  [smiles_d2[i]  for i in tr_d2])
X_feat_sht, sel_sht, sc_sht = fit_feature_pipeline(X_sht[tr_st], [smiles_sht[i] for i in tr_st])

sel = {'overlap': sel_ov, 'D2': sel_d2, '5HT2A': sel_sht}
sc  = {'overlap': sc_ov,  'D2': sc_d2,  '5HT2A': sc_sht}

print(f"  Overlap features: {X_feat_ov.shape[1]}")
print(f"  D2      features: {X_feat_d2.shape[1]}")
print(f"  5HT2A   features: {X_feat_sht.shape[1]}")

# Apply to val and test splits using the fitted pipeline
X_feat_ov_va  = build_feature_matrix(X_ov[va_ov],   [smiles_ov[i]  for i in va_ov],  sel_ov,  sc_ov)
X_feat_ov_te  = build_feature_matrix(X_ov[te_ov],   [smiles_ov[i]  for i in te_ov],  sel_ov,  sc_ov)
X_feat_d2_va  = build_feature_matrix(X_d2[va_d2],   [smiles_d2[i]  for i in va_d2],  sel_d2,  sc_d2)
X_feat_d2_te  = build_feature_matrix(X_d2[te_d2],   [smiles_d2[i]  for i in te_d2],  sel_d2,  sc_d2)
X_feat_sht_va = build_feature_matrix(X_sht[va_st],  [smiles_sht[i] for i in va_st],  sel_sht, sc_sht)
X_feat_sht_te = build_feature_matrix(X_sht[te_st],  [smiles_sht[i] for i in te_st],  sel_sht, sc_sht)

# Convenience: train slices are already built above
X_A_tr, y_A_tr = X_feat_ov,    y_del[tr_ov]
X_A_va, y_A_va = X_feat_ov_va, y_del[va_ov]
X_A_te, y_A_te = X_feat_ov_te, y_del[te_ov]

X_d2_tr, y_d2_tr = X_feat_d2,    y_d2[tr_d2]
X_d2_va, y_d2_va = X_feat_d2_va, y_d2[va_d2]
X_d2_te, y_d2_te = X_feat_d2_te, y_d2[te_d2]

X_st_tr, y_st_tr = X_feat_sht,    y_sht[tr_st]
X_st_va, y_st_va = X_feat_sht_va, y_sht[va_st]
X_st_te, y_st_te = X_feat_sht_te, y_sht[te_st]

print(f"Train: {len(X_A_tr):,}  Val: {len(X_A_va):,}  Test: {len(X_A_te):,}")

Fitting feature pipelines on training splits...
  Overlap features: 701
  D2      features: 794
  5HT2A   features: 761
Train: 1,901  Val: 407  Test: 407


## 2. Baseline tree models

Default hyperparameters — no tuning. Establishes where each family starts.

In [3]:
def eval_model(model, X_tr, y_tr, X_va, y_va, name=''):
    """Fit on train, evaluate on train and val only. Test is never touched here."""
    model.fit(X_tr, y_tr)
    results = {}
    for split, X, y in [('train', X_tr, y_tr), ('val', X_va, y_va)]:
        yp = model.predict(X)
        results[split] = dict(
            r2   = r2_score(y, yp),
            rmse = np.sqrt(mean_squared_error(y, yp)),
            mae  = mean_absolute_error(y, yp),
            pred = yp, true = y,
        )
    if name:
        print(f"  {name:28s}  train R²={results['train']['r2']:.3f}  "
              f"val R²={results['val']['r2']:.3f}")
    return results

TREE_DEFAULTS = dict(n_estimators=200, random_state=SEED, n_jobs=-1)

print("── Approach A (direct ΔpChEMBL) — baseline trees ──")
rf_A_base  = RandomForestRegressor(**TREE_DEFAULTS)
xgb_A_base = XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0)
lgb_A_base = LGBMRegressor(n_estimators=200, random_state=SEED, verbosity=-1)

res_rf_A  = eval_model(rf_A_base,  X_A_tr, y_A_tr, X_A_va, y_A_va, 'RF')
res_xgb_A = eval_model(xgb_A_base, X_A_tr, y_A_tr, X_A_va, y_A_va, 'XGBoost')
res_lgb_A = eval_model(lgb_A_base, X_A_tr, y_A_tr, X_A_va, y_A_va, 'LightGBM')

print("\n── Approach B (individual receptors) — baseline trees ──")
rf_B_d2   = RandomForestRegressor(**TREE_DEFAULTS)
rf_B_sht  = RandomForestRegressor(**TREE_DEFAULTS)
xgb_B_d2  = XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0)
xgb_B_sht = XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0)
lgb_B_d2  = LGBMRegressor(n_estimators=200, random_state=SEED, verbosity=-1)
lgb_B_sht = LGBMRegressor(n_estimators=200, random_state=SEED, verbosity=-1)

res_rf_Bd2   = eval_model(rf_B_d2,   X_d2_tr, y_d2_tr, X_d2_va, y_d2_va, 'RF D2')
res_rf_Bsht  = eval_model(rf_B_sht,  X_st_tr, y_st_tr, X_st_va, y_st_va, 'RF 5HT2A')
res_xgb_Bd2  = eval_model(xgb_B_d2,  X_d2_tr, y_d2_tr, X_d2_va, y_d2_va, 'XGB D2')
res_xgb_Bsht = eval_model(xgb_B_sht, X_st_tr, y_st_tr, X_st_va, y_st_va, 'XGB 5HT2A')
res_lgb_Bd2  = eval_model(lgb_B_d2,  X_d2_tr, y_d2_tr, X_d2_va, y_d2_va, 'LGB D2')
res_lgb_Bsht = eval_model(lgb_B_sht, X_st_tr, y_st_tr, X_st_va, y_st_va, 'LGB 5HT2A')

print("\n── Approach C (multi-task) — baseline trees ──")
# RF: native multi-output — each tree minimises combined impurity across both
# targets simultaneously. Genuine shared representation.
# XGB / LGB: no native multi-output support. Wrapped in MultiOutputRegressor
# which trains two independent forests. This IS Approach B with a different
# feature space. We include it for completeness but it is NOT a shared
# representation — the MLP two-headed architecture is the proper C for DL.
from sklearn.multioutput import MultiOutputRegressor

y_d2_ov  = merged['pChEMBL_D2'].values
y_sht_ov = merged['pChEMBL_5HT2A'].values
Y_tr     = np.column_stack([y_d2_ov[tr_ov], y_sht_ov[tr_ov]])
Y_va     = np.column_stack([y_d2_ov[va_ov], y_sht_ov[va_ov]])

rf_C_base  = RandomForestRegressor(**TREE_DEFAULTS)
xgb_C_base = MultiOutputRegressor(XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0))
lgb_C_base = MultiOutputRegressor(LGBMRegressor(n_estimators=200, random_state=SEED, verbosity=-1))

for model, name in [(rf_C_base,  'RF C (shared trees)'),
                    (xgb_C_base, 'XGB C (independent, wrapped)'),
                    (lgb_C_base, 'LGB C (independent, wrapped)')]:
    model.fit(X_A_tr, Y_tr)
    Y_tr_p = model.predict(X_A_tr)
    Y_va_p = model.predict(X_A_va)
    print(f"  {name:30s}  "
          f"train D2={r2_score(Y_tr[:, 0], Y_tr_p[:, 0]):.3f} "
          f"5HT2A={r2_score(Y_tr[:, 1], Y_tr_p[:, 1]):.3f}  |  "
          f"val D2={r2_score(Y_va[:, 0], Y_va_p[:, 0]):.3f} "
          f"5HT2A={r2_score(Y_va[:, 1], Y_va_p[:, 1]):.3f}")

── Approach A (direct ΔpChEMBL) — baseline trees ──
  RF                            train R²=0.957  val R²=0.554
  XGBoost                       train R²=0.989  val R²=0.537


KeyboardInterrupt: 

## 3. Tree baseline diagnostics

Residual plots and predicted vs actual for the direct model (Approach A, RF).
Reveal bias patterns, heteroscedasticity, and activity cliff failures.

In [ ]:
def diagnostic_plots(results, title):
    """Train and val only — test is never shown during model development."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (split, col_v) in zip(axes, [('train','#94A3B8'), ('val','#1D9E75')]):
        yp  = results[split]['pred']
        yt  = results[split]['true']
        ax.scatter(yt, yp, s=10, alpha=0.4, color=col_v, edgecolors='none')
        lim = max(abs(yt).max(), abs(yp).max()) + 0.3
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1, alpha=0.5)
        ax.set(xlabel='Actual ΔpChEMBL', ylabel='Predicted',
               title=f"{split}  R²={results[split]['r2']:.3f}  RMSE={results[split]['rmse']:.3f}",
               xlim=[-lim, lim], ylim=[-lim, lim])
        ax.set_aspect('equal'); ax.grid(alpha=0.2)
    plt.suptitle(title, fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"fig_02_diag_{title.replace(' ','_').lower()[:40]}.png",
                dpi=120, bbox_inches='tight')
    plt.show()

def residual_plot(results, title):
    """Residuals on val only."""
    fig, ax = plt.subplots(figsize=(7, 4))
    yp  = results['val']['pred']
    yt  = results['val']['true']
    res = yp - yt
    ax.scatter(yp, res, s=10, alpha=0.4, color='#1D9E75', edgecolors='none')
    ax.axhline(0, color='black', lw=1, linestyle='--')
    ax.set(xlabel='Predicted ΔpChEMBL', ylabel='Residual (pred − actual)',
           title=f"Val residuals — bias={res.mean():.3f}  std={res.std():.3f}")
    ax.grid(alpha=0.2)
    plt.suptitle(title, fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"fig_02_resid_{title.replace(' ','_').lower()[:40]}.png",
                dpi=120, bbox_inches='tight')
    plt.show()

TREE_DEFAULTS = dict(n_estimators=200, random_state=SEED, n_jobs=-1)

print("── Approach A (direct ΔpChEMBL) — baseline trees ──")
rf_A_base  = RandomForestRegressor(**TREE_DEFAULTS)
xgb_A_base = XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0)
lgb_A_base = LGBMRegressor(n_estimators=200, random_state=SEED, verbosity=-1)

res_rf_A  = eval_model(rf_A_base,  X_A_tr, y_A_tr, X_A_va, y_A_va, 'RF')
res_xgb_A = eval_model(xgb_A_base, X_A_tr, y_A_tr, X_A_va, y_A_va, 'XGBoost')
res_lgb_A = eval_model(lgb_A_base, X_A_tr, y_A_tr, X_A_va, y_A_va, 'LightGBM')

print("\n── Approach B (individual receptors) — baseline trees ──")
rf_B_d2  = RandomForestRegressor(**TREE_DEFAULTS)
rf_B_sht = RandomForestRegressor(**TREE_DEFAULTS)
res_rf_Bd2  = eval_model(rf_B_d2,  X_d2_tr, y_d2_tr, X_d2_va, y_d2_va, 'RF D2')
res_rf_Bsht = eval_model(rf_B_sht, X_st_tr, y_st_tr, X_st_va, y_st_va, 'RF 5HT2A')

## 4. Baseline MLP

Pyramid-down architecture fixed from HP search results (notebook 05):
`512 → 256 → 128`, GELU, BatchNorm, HuberLoss, cosine warm restarts.
Baseline uses sensible default hyperparameters before tuning.

In [ ]:
seed_everything(SEED)

# ── Architecture ──────────────────────────────────────────────────────────────
class PyramidMLP(nn.Module):
    """pyramid_down — best architecture from HP search."""
    def __init__(self, input_dim, dropout=0.25, n_outputs=1):
        super().__init__()
        dims = [512, 256, 128]
        layers, in_d = [], input_dim
        for h in dims:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h),
                       nn.GELU(), nn.Dropout(dropout)]
            in_d = h
        self.encoder = nn.Sequential(*layers)
        self.head     = nn.Linear(in_d, n_outputs)

    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(-1)


class TwoHeadMLP(nn.Module):
    """Shared encoder, two separate heads — Approach C multi-task."""
    def __init__(self, input_dim, dropout=0.25):
        super().__init__()
        shared = [512, 256]
        head_d = 128
        layers, in_d = [], input_dim
        for h in shared:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h),
                       nn.GELU(), nn.Dropout(dropout)]
            in_d = h
        self.encoder  = nn.Sequential(*layers)
        self.head_d2  = nn.Sequential(
            nn.Linear(in_d, head_d), nn.GELU(), nn.Linear(head_d, 1))
        self.head_sht = nn.Sequential(
            nn.Linear(in_d, head_d), nn.GELU(), nn.Linear(head_d, 1))

    def forward(self, x):
        h = self.encoder(x)
        return self.head_d2(h).squeeze(-1), self.head_sht(h).squeeze(-1)


# ── Training utilities ────────────────────────────────────────────────────────
def to_t(arr):
    return torch.tensor(arr, dtype=torch.float32).to(DEVICE)

def make_train_loader(dataset, batch_size, shuffle=True, generator=None):
    """DataLoader that avoids BatchNorm failures from size-1 batches."""
    n = len(dataset)
    bs = min(batch_size, n)
    drop_last = n > bs and (n % bs != 0)
    kwargs = dict(batch_size=bs, shuffle=shuffle, drop_last=drop_last)
    if generator is not None:
        kwargs['generator'] = generator
    return DataLoader(dataset, **kwargs)

def train_mlp(model, X_tr, y_tr, X_va, y_va,
              lr=0.00014, batch=64, wd=3e-5,
              epochs=300, patience=30, verbose=False, seed=SEED):
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched   = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)
    loss_fn = nn.HuberLoss(delta=1.0)
    g = torch.Generator(); g.manual_seed(seed)
    loader  = make_train_loader(
        TensorDataset(to_t(X_tr), to_t(y_tr)), batch, shuffle=True, generator=g)

    history = {'train': [], 'val': []}
    best_r2, best_state, wait = -np.inf, None, 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        sched.step()

        model.eval()
        with torch.no_grad():
            yp_tr = model(to_t(X_tr)).cpu().numpy()
            yp_va = model(to_t(X_va)).cpu().numpy()
        r2_tr = r2_score(y_tr, yp_tr)
        r2_va = r2_score(y_va, yp_va)
        history['train'].append(r2_tr)
        history['val'].append(r2_va)

        if r2_va > best_r2:
            best_r2 = r2_va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

        if verbose and epoch % 25 == 0:
            print(f"  epoch {epoch:3d}  train R²={r2_tr:.3f}  val R²={r2_va:.3f}")

    model.load_state_dict(best_state)
    return model, history


def train_mlp_multitask(model, X_tr, y_d2_tr, y_sht_tr,
                         X_va, y_d2_va, y_sht_va,
                         lr=0.00014, batch=64, wd=3e-5,
                         epochs=300, patience=30, seed=SEED):
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched   = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)
    loss_fn = nn.HuberLoss(delta=1.0)
    Y_tr    = np.column_stack([y_d2_tr, y_sht_tr])
    loader  = make_train_loader(
        TensorDataset(to_t(X_tr), to_t(Y_tr)), batch, shuffle=True)

    history = {'val_d2': [], 'val_sht': []}
    best_r2, best_state, wait = -np.inf, None, 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            out_d2, out_sht = model(xb)
            loss = loss_fn(out_d2, yb[:, 0]) + loss_fn(out_sht, yb[:, 1])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            pd2, psht = model(to_t(X_va))
        r2_d2  = r2_score(y_d2_va,  pd2.cpu().numpy())
        r2_sht = r2_score(y_sht_va, psht.cpu().numpy())
        history['val_d2'].append(r2_d2)
        history['val_sht'].append(r2_sht)

        mean_r2 = (r2_d2 + r2_sht) / 2
        if mean_r2 > best_r2:
            best_r2 = mean_r2
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    model.load_state_dict(best_state)
    return model, history


# ── Train baseline MLPs ───────────────────────────────────────────────────────
print("Training baseline MLPs (default hyperparameters)...")

# Approach A
torch.manual_seed(SEED)
mlp_A_base = PyramidMLP(X_A_tr.shape[1]).to(DEVICE)
mlp_A_base, hist_A = train_mlp(mlp_A_base, X_A_tr, y_A_tr, X_A_va, y_A_va)
mlp_A_base.eval()
with torch.no_grad():
    yp_A_va = mlp_A_base(to_t(X_A_va)).cpu().numpy()
print(f"  MLP A (direct)   val R²={r2_score(y_A_va, yp_A_va):.4f}")

# Approach B — D2
torch.manual_seed(SEED)
mlp_B_d2 = PyramidMLP(X_d2_tr.shape[1]).to(DEVICE)
mlp_B_d2, hist_Bd2 = train_mlp(mlp_B_d2, X_d2_tr, y_d2_tr, X_d2_va, y_d2_va)
mlp_B_d2.eval()
with torch.no_grad():
    yp_Bd2_va = mlp_B_d2(to_t(X_d2_va)).cpu().numpy()
print(f"  MLP B D2         val R²={r2_score(y_d2_va, yp_Bd2_va):.4f}")

# Approach B — 5HT2A
torch.manual_seed(SEED)
mlp_B_sht = PyramidMLP(X_st_tr.shape[1]).to(DEVICE)
mlp_B_sht, hist_Bsht = train_mlp(mlp_B_sht, X_st_tr, y_st_tr, X_st_va, y_st_va)
mlp_B_sht.eval()
with torch.no_grad():
    yp_Bsht_va = mlp_B_sht(to_t(X_st_va)).cpu().numpy()
print(f"  MLP B 5HT2A      val R²={r2_score(y_st_va, yp_Bsht_va):.4f}")

# Approach C — multi-task
# Uses overlap feature matrix for both receptors — same input for both heads
y_d2_ov  = merged['pChEMBL_D2'].values
y_sht_ov = merged['pChEMBL_5HT2A'].values

torch.manual_seed(SEED)
mlp_C = TwoHeadMLP(X_A_tr.shape[1]).to(DEVICE)
mlp_C, hist_C = train_mlp_multitask(
    mlp_C,
    X_A_tr, y_d2_ov[tr_ov], y_sht_ov[tr_ov],
    X_A_va, y_d2_ov[va_ov], y_sht_ov[va_ov])
mlp_C.eval()
with torch.no_grad():
    pd2_va, psht_va = mlp_C(to_t(X_A_va))
print(f"  MLP C D2         val R²={r2_score(y_d2_ov[va_ov], pd2_va.cpu().numpy()):.4f}")
print(f"  MLP C 5HT2A      val R²={r2_score(y_sht_ov[va_ov], psht_va.cpu().numpy()):.4f}")

## 5. MLP diagnostics — learning curves and residuals

In [ ]:
def plot_learning_curves(histories, labels, title):
    """Plot train and val R² curves. Solid = val, dashed = train."""
    fig, ax = plt.subplots(figsize=(10, 4))
    colors  = ['#1D9E75', '#D85A30', '#534AB7', '#D97706']
    for (key, hist), label, col in zip(histories.items(), labels, colors):
        val_curve   = hist.get('val',   hist.get('val_d2',  []))
        train_curve = hist.get('train', [])
        ax.plot(val_curve,   lw=1.8, color=col, label=f"{label} — val")
        if train_curve:
            ax.plot(train_curve, lw=1.2, color=col, linestyle='--',
                    alpha=0.5, label=f"{label} — train")
    ax.axhline(0.4, color='black', lw=1, linestyle=':', alpha=0.4, label='R²=0.4')
    ax.set(xlabel='Epoch', ylabel='R²', title=title)
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"fig_02_curves_{title.replace(' ','_').lower()[:30]}.png",
                dpi=120, bbox_inches='tight')
    plt.show()

# Learning curves
plot_learning_curves(
    {'A': hist_A, 'B_D2': hist_Bd2, 'B_5HT2A': hist_Bsht},
    ['MLP A (direct Δ)', 'MLP B — D2', 'MLP B — 5HT₂A'],
    'MLP learning curves — baseline')

# Approach A: predicted vs actual + residuals (val and test)
mlp_A_base.eval()
with torch.no_grad():
    yp_A_tr_arr = mlp_A_base(to_t(X_A_tr)).cpu().numpy()

res_mlp_A = {
    'train': {'pred': yp_A_tr_arr, 'true': y_A_tr,
               'r2': r2_score(y_A_tr, yp_A_tr_arr),
               'rmse': np.sqrt(mean_squared_error(y_A_tr, yp_A_tr_arr))},
    'val':   {'pred': yp_A_va, 'true': y_A_va,
               'r2': r2_score(y_A_va, yp_A_va),
               'rmse': np.sqrt(mean_squared_error(y_A_va, yp_A_va))},
}

diagnostic_plots(res_mlp_A, 'MLP A — Approach A (baseline)')
residual_plot(res_mlp_A,    'MLP A — Approach A (baseline)')

# Error breakdown by selectivity decile
fig, ax = plt.subplots(figsize=(10, 4))
df_diag = pd.DataFrame({'true': y_A_va, 'pred': yp_A_va,
                          'error': np.abs(yp_A_va - y_A_va)})
df_diag['decile'] = pd.qcut(df_diag['true'], 10, labels=False)
decile_err = df_diag.groupby('decile')['error'].mean()
decile_mid = df_diag.groupby('decile')['true'].mean()
ax.bar(decile_mid, decile_err, width=0.3, color='#534AB7', alpha=0.85, edgecolor='black')
ax.set(xlabel='Actual ΔpChEMBL (decile midpoint)', ylabel='Mean |error|',
       title='Error by selectivity decile — MLP A (val)')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('fig_02_decile_error.png', dpi=120, bbox_inches='tight')
plt.show()
print("Check: are errors larger at extreme selectivity values? (activity cliff region)")

## 6. Baseline summary

In [ ]:
print("BASELINE SUMMARY  (train / val only — test untouched until final eval)")
print("="*68)
print(f"{'Model':35s} | {'Train R²':>9s} | {'Val R²':>8s} | {'Val RMSE':>9s}")
print("-"*68)

mlp_A_base.eval(); mlp_B_d2.eval(); mlp_B_sht.eval(); mlp_C.eval()

def row(name, r2_tr, r2_va, rmse_va):
    tr = f"{r2_tr:.3f}" if r2_tr is not None else "—"
    rm = f"{rmse_va:.3f}" if rmse_va is not None else "—"
    print(f"  {name:33s} | {tr:>9s} | {r2_va:>8.3f} | {rm:>9s}")

# ── Approach A ────────────────────────────────────────────────────────────────
print("  Approach A — direct ΔpChEMBL")
for model, name in [(rf_A_base,'RF'), (xgb_A_base,'XGB'), (lgb_A_base,'LGB')]:
    row(f"  {name} A",
        r2_score(y_A_tr, model.predict(X_A_tr)),
        r2_score(y_A_va, model.predict(X_A_va)),
        np.sqrt(mean_squared_error(y_A_va, model.predict(X_A_va))))

with torch.no_grad():
    yp_tr = mlp_A_base(to_t(X_A_tr)).cpu().numpy()
    yp_va = mlp_A_base(to_t(X_A_va)).cpu().numpy()
row("  MLP A",
    r2_score(y_A_tr, yp_tr),
    r2_score(y_A_va, yp_va),
    np.sqrt(mean_squared_error(y_A_va, yp_va)))

# ── Approach B ────────────────────────────────────────────────────────────────
print("  Approach B — individual receptors")
for model, name, X_tr, y_tr, X_va, y_va in [
    (rf_B_d2,   'RF  B D2',    X_d2_tr, y_d2_tr, X_d2_va, y_d2_va),
    (xgb_B_d2,  'XGB B D2',   X_d2_tr, y_d2_tr, X_d2_va, y_d2_va),
    (lgb_B_d2,  'LGB B D2',   X_d2_tr, y_d2_tr, X_d2_va, y_d2_va),
    (rf_B_sht,  'RF  B 5HT2A', X_st_tr, y_st_tr, X_st_va, y_st_va),
    (xgb_B_sht, 'XGB B 5HT2A',X_st_tr, y_st_tr, X_st_va, y_st_va),
    (lgb_B_sht, 'LGB B 5HT2A',X_st_tr, y_st_tr, X_st_va, y_st_va),
]:
    row(f"  {name}",
        r2_score(y_tr, model.predict(X_tr)),
        r2_score(y_va, model.predict(X_va)),
        np.sqrt(mean_squared_error(y_va, model.predict(X_va))))

with torch.no_grad():
    pd2_tr = mlp_B_d2(to_t(X_d2_tr)).cpu().numpy()
    pd2_va = mlp_B_d2(to_t(X_d2_va)).cpu().numpy()
    pst_tr = mlp_B_sht(to_t(X_st_tr)).cpu().numpy()
    pst_va = mlp_B_sht(to_t(X_st_va)).cpu().numpy()
row("  MLP B D2",
    r2_score(y_d2_tr, pd2_tr), r2_score(y_d2_va, pd2_va),
    np.sqrt(mean_squared_error(y_d2_va, pd2_va)))
row("  MLP B 5HT2A",
    r2_score(y_st_tr, pst_tr), r2_score(y_st_va, pst_va),
    np.sqrt(mean_squared_error(y_st_va, pst_va)))

# ── Approach C ────────────────────────────────────────────────────────────────
print("  Approach C — multi-task")
for model, name in [(rf_C_base,'RF C (shared)'),
                    (xgb_C_base,'XGB C (indep.)'),
                    (lgb_C_base,'LGB C (indep.)')]:
    Y_tr_p = model.predict(X_A_tr)
    Y_va_p = model.predict(X_A_va)
    row(f"  {name} D2",
        r2_score(Y_tr[:, 0], Y_tr_p[:, 0]),
        r2_score(Y_va[:, 0], Y_va_p[:, 0]),
        np.sqrt(mean_squared_error(Y_va[:, 0], Y_va_p[:, 0])))
    row(f"  {name} 5HT2A",
        r2_score(Y_tr[:, 1], Y_tr_p[:, 1]),
        r2_score(Y_va[:, 1], Y_va_p[:, 1]),
        np.sqrt(mean_squared_error(Y_va[:, 1], Y_va_p[:, 1])))

with torch.no_grad():
    pd2_tr, pst_tr = mlp_C(to_t(X_A_tr))
    pd2_va, pst_va = mlp_C(to_t(X_A_va))
row("  MLP C D2",
    r2_score(y_d2_ov[tr_ov], pd2_tr.cpu().numpy()),
    r2_score(y_d2_ov[va_ov], pd2_va.cpu().numpy()),
    np.sqrt(mean_squared_error(y_d2_ov[va_ov], pd2_va.cpu().numpy())))
row("  MLP C 5HT2A",
    r2_score(y_sht_ov[tr_ov], pst_tr.cpu().numpy()),
    r2_score(y_sht_ov[va_ov], pst_va.cpu().numpy()),
    np.sqrt(mean_squared_error(y_sht_ov[va_ov], pst_va.cpu().numpy())))

print("\nTest set evaluation happens once in the final cell, after all tuning.")

## 7. Tuning

Two steps, run in order:

1. **Trees** — Optuna search over RF / XGB / LGB (`N_TRIALS` per task)
2. **MLP** — fixed PyramidMLP; tune dropout, lr, weight decay, batch size on Approach A, then transfer hyperparameters to B and C

Baselines in §2–§6 are diagnostic only; the saved `models.pkl` uses the tuned models below.

In [ ]:
seed_everything(SEED)

N_TRIALS = 30   # overnight run — ~6-8h for trees + MLP across all tasks


def _tree_params(params, prefix):
    """Strip model prefix from Optuna param names for sklearn/xgb/lgb constructors."""
    plen = len(prefix)
    return {k[plen:]: v for k, v in params.items() if k.startswith(prefix)}


def tune_tree(X_tr, y_tr, X_va, y_va, n_trials=N_TRIALS, seed=SEED):
    """
    Rich parameter space for overnight run.
    Each model family gets its full set of meaningful hyperparameters.
    Pruner kills bad trials early to keep the search efficient.

    Hyperparameters are prefixed (rf_/xgb_/lgb_) so Optuna never mixes
    incompatible distributions for the same name across model branches.
    """
    def obj(trial):
        mtype = trial.suggest_categorical('model', ['rf', 'xgb', 'lgb'])

        if mtype == 'rf':
            m = RandomForestRegressor(
                n_estimators      = trial.suggest_int('rf_n_estimators', 100, 1000),
                max_depth         = trial.suggest_int('rf_max_depth', 3, 20),
                min_samples_leaf  = trial.suggest_int('rf_min_samples_leaf', 1, 30),
                min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 20),
                max_features      = trial.suggest_float('rf_max_features', 0.05, 0.5),
                max_samples       = trial.suggest_float('rf_max_samples', 0.5, 1.0),
                bootstrap         = True,
                random_state=seed, n_jobs=-1)

        elif mtype == 'xgb':
            m = XGBRegressor(
                n_estimators      = trial.suggest_int('xgb_n_estimators', 100, 1000),
                max_depth         = trial.suggest_int('xgb_max_depth', 2, 10),
                max_leaves        = trial.suggest_int('xgb_max_leaves', 0, 64),
                learning_rate     = trial.suggest_float('xgb_learning_rate', 0.005, 0.3, log=True),
                subsample         = trial.suggest_float('xgb_subsample', 0.4, 1.0),
                colsample_bytree  = trial.suggest_float('xgb_colsample_bytree', 0.2, 1.0),
                colsample_bylevel = trial.suggest_float('xgb_colsample_bylevel', 0.2, 1.0),
                min_child_weight  = trial.suggest_int('xgb_min_child_weight', 1, 30),
                gamma             = trial.suggest_float('xgb_gamma', 0.0, 5.0),
                reg_alpha         = trial.suggest_float('xgb_reg_alpha', 1e-5, 50.0, log=True),
                reg_lambda        = trial.suggest_float('xgb_reg_lambda', 1e-5, 50.0, log=True),
                grow_policy       = trial.suggest_categorical('xgb_grow_policy',
                                        ['depthwise', 'lossguide']),
                random_state=seed, verbosity=0, n_jobs=-1)

        else:  # lgb
            m = LGBMRegressor(
                n_estimators      = trial.suggest_int('lgb_n_estimators', 100, 1000),
                num_leaves        = trial.suggest_int('lgb_num_leaves', 8, 256),
                max_depth         = trial.suggest_int('lgb_max_depth', -1, 20),
                learning_rate     = trial.suggest_float('lgb_learning_rate', 0.005, 0.3, log=True),
                subsample         = trial.suggest_float('lgb_subsample', 0.4, 1.0),
                subsample_freq    = trial.suggest_int('lgb_subsample_freq', 1, 10),
                colsample_bytree  = trial.suggest_float('lgb_colsample_bytree', 0.2, 1.0),
                min_child_samples = trial.suggest_int('lgb_min_child_samples', 5, 100),
                min_child_weight  = trial.suggest_float('lgb_min_child_weight', 1e-4, 0.1, log=True),
                reg_alpha         = trial.suggest_float('lgb_reg_alpha', 1e-5, 50.0, log=True),
                reg_lambda        = trial.suggest_float('lgb_reg_lambda', 1e-5, 50.0, log=True),
                extra_trees       = trial.suggest_categorical('lgb_extra_trees', [True, False]),
                path_smooth       = trial.suggest_float('lgb_path_smooth', 0.0, 1.0),
                random_state=seed, verbosity=-1, n_jobs=-1)

        m.fit(X_tr, y_tr)
        return r2_score(y_va, m.predict(X_va))

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed, n_startup_trials=30,
                                            multivariate=True,
                                            group=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=20,
                                            n_warmup_steps=0))
    study.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    return study


def refit_best_tree(study, X_tr, y_tr):
    p     = study.best_params.copy()
    mtype = p.pop('model')
    kwargs = _tree_params(p, f'{mtype}_')
    if mtype == 'rf':
        kwargs.setdefault('bootstrap', True)
        m = RandomForestRegressor(**kwargs, random_state=SEED, n_jobs=-1)
    elif mtype == 'xgb':
        m = XGBRegressor(**kwargs, random_state=SEED, verbosity=0, n_jobs=-1)
    else:
        m = LGBMRegressor(**kwargs, random_state=SEED, verbosity=-1, n_jobs=-1)
    m.fit(X_tr, y_tr)
    return m


# ── Run overnight search — trees ──────────────────────────────────────────────
tree_tasks = [
    ('A — direct Δ',  X_A_tr,  y_A_tr,           X_A_va,  y_A_va),
    ('B — D2',        X_d2_tr, y_d2_tr,           X_d2_va, y_d2_va),
    ('B — 5HT2A',     X_st_tr, y_st_tr,           X_st_va, y_st_va),
    ('C — D2',        X_A_tr,  y_d2_ov[tr_ov],    X_A_va,  y_d2_ov[va_ov]),
    ('C — 5HT2A',     X_A_tr,  y_sht_ov[tr_ov],   X_A_va,  y_sht_ov[va_ov]),
]

tree_studies = {}
tree_best    = {}

for label, X_tr, y_tr, X_va, y_va in tree_tasks:
    print(f"\n── Trees: Approach {label} ({N_TRIALS} trials) ──")
    study = tune_tree(X_tr, y_tr, X_va, y_va)
    model = refit_best_tree(study, X_tr, y_tr)
    tree_studies[label] = study
    tree_best[label]    = model
    val_r2 = r2_score(y_va, model.predict(X_va))
    tr_r2  = r2_score(y_tr, model.predict(X_tr))
    print(f"  Best ({study.best_params['model'].upper()})  "
          f"train={tr_r2:.4f}  val={val_r2:.4f}")

print("\n── Tuned tree summary ──────────────────────────────────────")
print(f"{'Task':22s} | {'Train R²':>9s} | {'Val R²':>8s}")
print("-"*46)
for label, X_tr, y_tr, X_va, y_va in tree_tasks:
    m = tree_best[label]
    print(f"  {label:20s} | {r2_score(y_tr, m.predict(X_tr)):>9.4f} | "
          f"{r2_score(y_va, m.predict(X_va)):>8.4f}")

# Aliases used by save / downstream notebooks
best_tree_A = tree_best['A — direct Δ']
study_tree  = tree_studies['A — direct Δ']


### 7b. MLP tuning

The next cell tunes a **fixed PyramidMLP** architecture (512→256→128, GELU, BatchNorm) — dropout, learning rate, weight decay, and batch size only. This matches the baseline architecture and is what downstream notebooks expect.

> **Note:** An extended architecture search (`tune_mlp_full` / `tune_mlp_equal`) was removed from this notebook to avoid training the same tasks three times. Use `02_models.ipynb` for full architecture search if needed.

In [ ]:
if 'N_TRIALS' not in globals():
    N_TRIALS = 30   # tree tuning cell may override

def tune_mlp(X_tr, y_tr, X_va, y_va, n_trials=N_TRIALS, seed=SEED):
    """
    Architecture fixed (pyramid_down 512→256→128, GELU, BatchNorm, warm restarts).
    Tune only the regularisation and optimisation hyperparameters.

    Informed by diagnostics:
    - MLP train/val gap smaller than trees but still present → increase dropout upper bound
    - Default lr=0.00014 worked well → keep narrow range around it
    - Smaller batches (64) gave better generalisation in prior search
    - weight_decay is underexplored → widen range
    """
    def obj(trial):
        dropout = trial.suggest_float('dropout', 0.10, 0.50)
        lr      = trial.suggest_float('lr', 5e-5, 2e-3, log=True)
        wd      = trial.suggest_float('wd', 1e-5, 5e-2, log=True)
        batch   = trial.suggest_categorical('batch', [32, 64, 128, 256])

        torch.manual_seed(seed)
        m = PyramidMLP(X_tr.shape[1], dropout=dropout).to(DEVICE)
        _, h = train_mlp(m, X_tr, y_tr, X_va, y_va,
                         lr=lr, wd=wd, batch=batch,
                         epochs=200, patience=25)
        return max(h['val'])

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed, n_startup_trials=10))
    # Seed with known good config so TPE starts from a strong point
    study.enqueue_trial({'dropout': 0.25, 'lr': 0.00014, 'wd': 3e-5, 'batch': 64})
    study.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    return study

print(f"\nTuning MLP — Approach A ({N_TRIALS} trials)...")
study_mlp_A = tune_mlp(X_A_tr, y_A_tr, X_A_va, y_A_va)
bp          = study_mlp_A.best_params
print(f"Best val R²: {study_mlp_A.best_value:.4f}  params: {bp}")

# Refit best MLP with more epochs now that HP are fixed
torch.manual_seed(SEED)
best_mlp_A = PyramidMLP(X_A_tr.shape[1], dropout=bp['dropout']).to(DEVICE)
best_mlp_A, hist_A_tuned = train_mlp(
    best_mlp_A, X_A_tr, y_A_tr, X_A_va, y_A_va,
    lr=bp['lr'], wd=bp['wd'], batch=bp['batch'],
    epochs=300, patience=40)

best_mlp_A.eval()
with torch.no_grad():
    yp_va = best_mlp_A(to_t(X_A_va)).cpu().numpy()
    yp_tr = best_mlp_A(to_t(X_A_tr)).cpu().numpy()
print(f"Tuned MLP A — train R²={r2_score(y_A_tr, yp_tr):.4f}  "
      f"val R²={r2_score(y_A_va, yp_va):.4f}")

# Apply same HP to B and C (transfer: same architecture, same data scale)
print("\nTraining tuned MLP B and C with same hyperparameters...")
torch.manual_seed(SEED)
best_mlp_B_d2 = PyramidMLP(X_d2_tr.shape[1], dropout=bp['dropout']).to(DEVICE)
best_mlp_B_d2, _ = train_mlp(
    best_mlp_B_d2, X_d2_tr, y_d2_tr, X_d2_va, y_d2_va,
    lr=bp['lr'], wd=bp['wd'], batch=bp['batch'], epochs=300, patience=40)

torch.manual_seed(SEED)
best_mlp_B_sht = PyramidMLP(X_st_tr.shape[1], dropout=bp['dropout']).to(DEVICE)
best_mlp_B_sht, _ = train_mlp(
    best_mlp_B_sht, X_st_tr, y_st_tr, X_st_va, y_st_va,
    lr=bp['lr'], wd=bp['wd'], batch=bp['batch'], epochs=300, patience=40)

torch.manual_seed(SEED)
best_mlp_C = TwoHeadMLP(X_A_tr.shape[1], dropout=bp['dropout']).to(DEVICE)
best_mlp_C, hist_C_tuned = train_mlp_multitask(
    best_mlp_C,
    X_A_tr, y_d2_ov[tr_ov], y_sht_ov[tr_ov],
    X_A_va, y_d2_ov[va_ov], y_sht_ov[va_ov],
    lr=bp['lr'], wd=bp['wd'], batch=bp['batch'], epochs=300, patience=40)

for m in [best_mlp_B_d2, best_mlp_B_sht, best_mlp_C]:
    m.eval()

with torch.no_grad():
    r2_Bd2  = r2_score(y_d2_va,          best_mlp_B_d2(to_t(X_d2_va)).cpu().numpy())
    r2_Bsht = r2_score(y_st_va,          best_mlp_B_sht(to_t(X_st_va)).cpu().numpy())
    pd2_va, pst_va = best_mlp_C(to_t(X_A_va))
    r2_Cd2  = r2_score(y_d2_ov[va_ov],  pd2_va.cpu().numpy())
    r2_Csht = r2_score(y_sht_ov[va_ov], pst_va.cpu().numpy())

print(f"  MLP B D2     val R²={r2_Bd2:.4f}")
print(f"  MLP B 5HT2A  val R²={r2_Bsht:.4f}")
print(f"  MLP C D2     val R²={r2_Cd2:.4f}")
print(f"  MLP C 5HT2A  val R²={r2_Csht:.4f}")

## 8. Post-tuning diagnostics

In [ ]:
# Learning curve: baseline vs tuned
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_A['val'],       lw=1.5, color='#94A3B8', alpha=0.7, label='MLP A baseline — val')
ax.plot(hist_A['train'],     lw=1.0, color='#94A3B8', alpha=0.4, linestyle='--', label='MLP A baseline — train')
ax.plot(hist_A_tuned['val'], lw=2.0, color='#1D9E75', label='MLP A tuned — val')
ax.plot(hist_A_tuned['train'], lw=1.2, color='#1D9E75', alpha=0.5, linestyle='--', label='MLP A tuned — train')
ax.axhline(res_rf_A['val']['r2'], color='#D85A30', lw=1.5,
           linestyle=':', label=f"RF baseline val R²={res_rf_A['val']['r2']:.3f}")
ax.set(xlabel='Epoch', ylabel='R²', title='Learning curves — baseline vs tuned MLP A')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_02_curves_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# Post-tuning diagnostics — train and val only
best_mlp_A.eval()
with torch.no_grad():
    yp_tr = best_mlp_A(to_t(X_A_tr)).cpu().numpy()
    yp_va = best_mlp_A(to_t(X_A_va)).cpu().numpy()

res_mlp_A_tuned = {
    'train': {'pred': yp_tr, 'true': y_A_tr,
               'r2':   r2_score(y_A_tr, yp_tr),
               'rmse': np.sqrt(mean_squared_error(y_A_tr, yp_tr))},
    'val':   {'pred': yp_va, 'true': y_A_va,
               'r2':   r2_score(y_A_va, yp_va),
               'rmse': np.sqrt(mean_squared_error(y_A_va, yp_va))},
}

diagnostic_plots(res_mlp_A_tuned, 'MLP A — tuned')
residual_plot(res_mlp_A_tuned,    'MLP A — tuned')

## 9. Save

In [ ]:
models = {
    # Tree models
    'best_tree_A':    best_tree_A,
    'tree_study_A':   study_tree,
    'tree_studies':   tree_studies,
    'tree_best':      tree_best,
    # MLP — Approach A
    'mlp_A':          best_mlp_A,
    'mlp_A_base':     mlp_A_base,
    'mlp_params':     bp,
    'study_mlp_A':    study_mlp_A,
    # MLP — Approach B
    'mlp_B_d2':       best_mlp_B_d2,
    'mlp_B_sht':      best_mlp_B_sht,
    # MLP — Approach C
    'mlp_C':          best_mlp_C,
    # Selectors / scalers (needed by downstream notebooks)
    'selectors':      sel,
    'scalers':        sc,
    # Split indices
    'splits':         sp,
}

with open('models.pkl', 'wb') as f:
    pickle.dump(models, f)

print("Saved: models.pkl")
print(f"  Keys: {list(models.keys())}")


## 9. Final test set evaluation

Test set touched **once**, after all tuning decisions are complete.
These numbers are reported in the paper.

In [ ]:
print("FINAL TEST SET EVALUATION")
print("="*60)
print(f"{'Model':30s} | {'Val R²':>8s} | {'Test R²':>8s}")
print("-"*60)

best_mlp_A.eval(); best_mlp_B_d2.eval(); best_mlp_B_sht.eval(); best_mlp_C.eval()

with torch.no_grad():
    # Tree models
    for model, name, X_va, y_va, X_te, y_te in [
        (best_tree_A, 'Best tree A (direct)', X_A_va, y_A_va, X_A_te, y_A_te),
    ]:
        print(f"  {name:28s} | {r2_score(y_va, model.predict(X_va)):>8.3f} | "
              f"{r2_score(y_te, model.predict(X_te)):>8.3f}")

    # MLP A
    yp_va = best_mlp_A(to_t(X_A_va)).cpu().numpy()
    yp_te = best_mlp_A(to_t(X_A_te)).cpu().numpy()
    print(f"  {'MLP A (direct)':28s} | {r2_score(y_A_va, yp_va):>8.3f} | "
          f"{r2_score(y_A_te, yp_te):>8.3f}")

    # MLP B — derive Δ on test set
    pd2_te  = best_mlp_B_d2(to_t(X_d2_te)).cpu().numpy()
    psht_te = best_mlp_B_sht(to_t(X_st_te)).cpu().numpy()
    print(f"  {'MLP B D2':28s} | {r2_score(y_d2_va, best_mlp_B_d2(to_t(X_d2_va)).cpu().numpy()):>8.3f} | "
          f"{r2_score(y_d2_te, pd2_te):>8.3f}")
    print(f"  {'MLP B 5HT2A':28s} | {r2_score(y_st_va, best_mlp_B_sht(to_t(X_st_va)).cpu().numpy()):>8.3f} | "
          f"{r2_score(y_st_te, psht_te):>8.3f}")

    # MLP C
    pd2_te, psht_te = best_mlp_C(to_t(X_A_te))
    pd2_va, psht_va = best_mlp_C(to_t(X_A_va))
    print(f"  {'MLP C D2 (multi-task)':28s} | {r2_score(y_d2_ov[va_ov], pd2_va.cpu().numpy()):>8.3f} | "
          f"{r2_score(y_d2_ov[te_ov], pd2_te.cpu().numpy()):>8.3f}")
    print(f"  {'MLP C 5HT2A (multi-task)':28s} | {r2_score(y_sht_ov[va_ov], psht_va.cpu().numpy()):>8.3f} | "
          f"{r2_score(y_sht_ov[te_ov], psht_te.cpu().numpy()):>8.3f}")